# Fine-tune anhone3/LiteALPR (SVTR26-Tiny) cho Biển số Xe máy Việt Nam

Notebook này huấn luyện tiếp (fine-tune) từ trọng số gốc `svtr26_tiny/best.pth` trên bộ dữ liệu đã làm sạch `tap_finetune_svtr26.zip`.

**Lưu ý quan trọng:** Toàn bộ checkpoint (`best.pth`, `best.onnx`) được lưu trực tiếp lên **Google Drive** để không bao giờ bị mất khi Colab ngắt kết nối.

In [ ]:
# Bước 1: Kết nối Google Drive và kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
# Bước 2: Tải mã nguồn LiteALPR và cài đặt thư viện cần thiết
%cd /content
!git clone https://github.com/vn-anhnth/LiteALPR.git
%cd /content/LiteALPR
!pip install -q ultralytics imgaug pyclipper onnxscript onnxslim lmdb rapidfuzz onnx onnxruntime-gpu huggingface_hub pyyaml tqdm pillow opencv-python
# Vá lỗi tương thích imgaug 0.4.0 với NumPy 2.0+ trên Colab
!python -c "import pathlib, imgaug; p = pathlib.Path(imgaug.__file__).with_name('imgaug.py'); s = p.read_text(); shim = 'if not hasattr(np, \"sctypes\"): np.sctypes = {\"int\":[np.int8,np.int16,np.int32,np.int64],\"uint\":[np.uint8,np.uint16,np.uint32,np.uint64],\"float\":[np.float16,np.float32,np.float64],\"complex\":[np.complex64,np.complex128],\"others\":[bool,object,bytes,str,np.void]}\n'; p.write_text(s.replace('import numpy as np\n', 'import numpy as np\n' + shim, 1)) if 'hasattr(np, "sctypes")' not in s else None; print('Patched imgaug OK!')"

In [ ]:
# Bước 3: Giải nén bộ dữ liệu tap_finetune_svtr26.zip (bạn tải file zip lên Google Drive trước)
# Sửa đường dẫn bên dưới đúng nơi bạn lưu tap_finetune_svtr26.zip trên Google Drive
DUONG_DAN_ZIP = '/content/drive/MyDrive/tap_finetune_svtr26.zip'

!mkdir -p /content/du_lieu_vn
!unzip -q -o "{DUONG_DAN_ZIP}" -d /content/du_lieu_vn
!ls -lh /content/du_lieu_vn
!head -n 5 /content/du_lieu_vn/train_labels.txt

In [ ]:
# Bước 4: Đóng gói dữ liệu sang định dạng LMDB mà LiteALPR sử dụng khi huấn luyện
%cd /content/LiteALPR
!python tools/create_lmdb_dataset.py \
  --data_dir /content/du_lieu_vn \
  --label_files train_labels.txt val_labels.txt test_labels.txt \
  --output_dir /content/lmdb_vn
!ls -lh /content/lmdb_vn

In [ ]:
# Bước 5: Tải trọng số gốc svtr26_tiny/best.pth từ Hugging Face
from huggingface_hub import hf_hub_download
import shutil

duong_dan_goc = hf_hub_download(repo_id='anhone3/LiteALPR', filename='svtr26_tiny/best.pth')
shutil.copy(duong_dan_goc, '/content/svtr26_tiny_pretrained.pth')
print('Đã tải xong trọng số gốc:', '/content/svtr26_tiny_pretrained.pth')

In [ ]:
# Bước 6: Fine-tune SVTR26-Tiny (lưu toàn bộ checkpoint trực tiếp vào Google Drive)
THU_MUC_LUU_DRIVE = '/content/drive/MyDrive/LITEALPR_FINETUNE_VN'
!mkdir -p "{THU_MUC_LUU_DRIVE}"

%cd /content/LiteALPR
!python tools/train_rec.py \
  -c configs/rec/svtr26/svtr26_tiny.yml \
  -o Global.pretrained_model=/content/svtr26_tiny_pretrained.pth \
     Global.output_dir="{THU_MUC_LUU_DRIVE}" \
     Global.epoch_num=30 \
     Global.eval_epoch_step="[0, 1]" \
     LRScheduler.warmup_epoch=3 \
     Optimizer.lr=0.0002 \
     Train.loader.batch_size_per_card=64 \
     Train.sampler.first_bs=64 \
     Eval.loader.batch_size_per_card=64 \
     Eval.sampler.first_bs=64 \
     Train.dataset.data_dir_list="['/content/lmdb_vn/train']" \
     Eval.dataset.data_dir_list="['/content/lmdb_vn/val']"

In [ ]:
# Bước 7: Đánh giá trên tập Test (biển số hoàn toàn mới) và xuất ra file ONNX
%cd /content/LiteALPR
!python tools/eval_rec.py \
  -c configs/rec/svtr26/svtr26_tiny.yml \
  -o Global.pretrained_model="{THU_MUC_LUU_DRIVE}/best.pth" \
     Eval.loader.batch_size_per_card=64 \
     Eval.sampler.first_bs=64 \
     Eval.dataset.data_dir_list="['/content/lmdb_vn/test']"

!python tools/export_rec.py \
  -m "{THU_MUC_LUU_DRIVE}/best.pth" \
  --save_path "{THU_MUC_LUU_DRIVE}/onnx_export/best.onnx" \
  --dynamic

!ls -lh "{THU_MUC_LUU_DRIVE}/onnx_export"